# Load processed data + merged temperature

In [12]:
import os
from pathlib import Path

import pandas as pd
from sklearn.feature_selection import mutual_info_regression

from energy_forecast.features.calendar import create_calendar_features, add_cyclical_encoding
from energy_forecast.features.lags import create_day_ahead_features

In [13]:
if os.getcwd().endswith("notebooks"):
    os.chdir("..")
df = pd.read_csv("data/processed/demand_with_temperature.csv", index_col=0, parse_dates=True)
# Saved in UTC; calendar features need local (Europe/London) time
df.index = df.index.tz_convert("Europe/London")
df

,settlement_period,nd,embedded_wind_generation,embedded_wind_capacity,embedded_solar_generation,embedded_solar_capacity,non_bm_stor,pump_storage_pumping,ifa_flow,ifa2_flow,britned_flow,moyle_flow,east_west_flow,nemo_flow,is_holiday,temperature
settlement_date,,,,,,,,,,,,,,,,
2009-01-01 00:00:00+00:00,1,37910,54,1403,0,0,0,33,2002,0,0,-161,0,0,1,-1.604160
2009-01-01 00:30:00+00:00,2,38047,53,1403,0,0,0,157,2002,0,0,-160,0,0,1,-1.539581
2009-01-01 01:00:00+00:00,3,37380,53,1403,0,0,0,511,2002,0,0,-160,0,0,1,-1.475002
2009-01-01 01:30:00+00:00,4,36426,50,1403,0,0,0,589,1772,0,0,-160,0,0,1,-1.541587
2009-01-01 02:00:00+00:00,5,35687,50,1403,0,0,0,851,1753,0,0,-160,0,0,1,-1.608172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-02-08 21:30:00+00:00,44,30670,3855,6562,0,15905,0,12,2000,-4,439,-452,-531,999,0,6.090917
2024-02-08 22:00:00+00:00,45,28684,3873,6562,0,15905,0,12,1999,-4,141,-451,-531,897,0,6.224113
2024-02-08 22:30:00+00:00,46,27147,3890,6562,0,15905,0,112,2000,-4,139,-452,-531,893,0,6.311527


# Correlation Check — Kendall vs. Mutual Information

### Kendall

In [14]:
df_diff = df.diff()

# Kendall on levels picks up shared long-term trends; on differences it measures short-term
# co-movement, which is what mutual information (below) is computed on as well
kendall = pd.DataFrame({
    "levels": df.corr(method="kendall", numeric_only=True)["nd"],
    "differences": df_diff.corr(method="kendall", numeric_only=True)["nd"],
}).drop(index="nd")
kendall.sort_values("differences", key=abs, ascending=False)

,levels,differences
pump_storage_pumping,-0.289547,-0.236053
temperature,-0.170460,0.163604
ifa_flow,0.015754,0.136236
non_bm_stor,0.164895,0.107964
settlement_period,0.290570,0.103870
nemo_flow,-0.075224,0.102330
britned_flow,0.075273,0.094797
moyle_flow,-0.221476,-0.091861
east_west_flow,-0.156772,-0.082142
ifa2_flow,0.011234,0.078844


### Mutual information

In [15]:
X = df_diff.drop(columns=['nd']).dropna()
y = df_diff.loc[X.index, 'nd']

mi_scores = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)
mi_series

embedded_solar_generation    0.216276
pump_storage_pumping         0.157230
temperature                  0.083732
ifa_flow                     0.050719
moyle_flow                   0.035068
britned_flow                 0.029893
east_west_flow               0.027735
nemo_flow                    0.026350
non_bm_stor                  0.018671
settlement_period            0.017448
embedded_wind_generation     0.014740
ifa2_flow                    0.013966
embedded_solar_capacity      0.004584
is_holiday                   0.000209
embedded_wind_capacity       0.000000
dtype: float64

Kendall correlation is computed twice: on **levels**, where shared long-term trends dominate,
and on **differences**, which measures short-term co-movement. Mutual information is computed on
differences, so the like-for-like comparison of the two methods is Kendall (differences) vs
mutual information; the levels column shows which relationships are mostly trend.

Mutual information ranks `embedded_solar_generation` as the strongest relevant feature
(0.216), while Kendall almost entirely misses it, both on levels (0.067) and on differences
(0.002). This is consistent with solar generation's relationship to demand being confounded with
time-of-day rather than monotonic, and supports retaining solar generation as a feature, in
contrast to the original thesis's approach of dropping it based on a limited OLS test.
`embedded_wind_generation`, by contrast, shows almost no relationship on either measure
(Kendall on differences −0.006, MI 0.015). It is still retained, since its demand-suppressing
effect may only show in combination with other features, but this check does not support it on
its own; MRFO and SHAP decide whether it is useful.

`embedded_solar_capacity` and `embedded_wind_capacity` have the strongest Kendall correlations
on levels (−0.299 and −0.297, narrowly ahead of `settlement_period` and `pump_storage_pumping`),
but near-zero mutual information (0.005 and 0.000) and small Kendall on differences (−0.044 and
−0.008). This confirms these correlations are a trend artifact rather than a genuine
short-term relationship, since capacity changes infrequently rather than daily.

`pump_storage_pumping` (Kendall on differences −0.236, MI 0.157) and `temperature` (0.164, MI
0.084) show consistent, moderate relevance across both methods and are retained with confidence.
Note that temperature's Kendall sign flips between levels (−0.170: cold weather means higher
demand) and differences (+0.164): the differenced relationship mostly reflects the shared daily
cycle (both rise during the day) rather than the heating effect.

Among the interconnectors, `ifa_flow` is the most relevant on both measures (Kendall on
differences 0.136, MI 0.051), which supports keeping it.

`settlement_period` shows low mutual information (0.017) despite known time-of-day effects on
demand (established via the boxplots and MSTL decomposition earlier). This is likely a
limitation of applying mutual information to differenced cyclical data rather than evidence
that time-of-day is unimportant; time-of-day will instead be captured via cyclical (sine/cosine)
encodings in the feature engineering step below, rather than relying on this correlation check
alone.

**Features retained going forward:** temperature, pump_storage_pumping, non_bm_stor,
embedded_solar_generation, embedded_wind_generation, is_holiday, and all six interconnector
flow columns (`ifa_flow`, `ifa2_flow`, `britned_flow`, `moyle_flow`, `east_west_flow`,
`nemo_flow`; mostly weak individually but potentially useful in combination, to be confirmed via
SHAP). `embedded_solar_capacity` and `embedded_wind_capacity` are dropped, as their correlation
is a trend artifact rather than genuine signal.

In [16]:
df = df.drop(columns=["embedded_solar_capacity", "embedded_wind_capacity"])

# Feature Construction

With the correlation analysis complete, the following cells build the actual
feature set to be used for modelling. Calendar, cyclical, lag, and rolling
features are all implemented as reusable functions in
`src/energy_forecast/features/`, rather than written inline, so they can be
reused consistently across notebooks and in the final training pipeline.

- **Calendar features** (`create_calendar_features`) provide the basic date/time
  components (hour, day of week, month, etc.) established as relevant in the
  earlier EDA (via boxplots and MSTL seasonality). `year` is created but left out
  of the saved feature set: the test period (2021–2024) lies beyond the training
  years, and models can't extrapolate it reliably (trees treat every later year as
  the last one seen). `year` would likely look important when fitting the training
  data, since GB demand fell steadily over 2009–2021 and `year` is a clean proxy for
  that trend, but that reliance does not carry over to unseen later years. The
  current demand level is instead captured by the lag and rolling features, which
  adapt automatically when demand shifts (e.g. the 2022 energy-crisis drop).
- **Cyclical encoding** (`add_cyclical_encoding`) converts the cyclical calendar
  columns (hour, day of week, month) into sine/cosine pairs, avoiding the
  artificial discontinuity of raw integer encoding at wraparound points (e.g.
  hour 23 to hour 0).
All demand-based and grid features are built by `create_day_ahead_features` in
`src/energy_forecast/features/lags.py`, which is unit-tested to use only information at least
48 steps old.

- **Lag features** (`create_lag_features`) are built at lags 48, 288 and 336. The
  ACF/PACF analysis in the previous notebook identified lags 1, 2, 48, 288 and 336 as
  carrying independent signal, but this project forecasts **day-ahead**
  (horizon h = 48 half-hours, direct strategy): to predict demand at time t, only
  demand observed up to t − 48 may be used. Lags 1 and 2 would not be available
  a day in advance, so only the lags ≥ 48 are kept.
- **Rolling statistics** (`create_rolling_features`) add 1-day and 1-week rolling
  mean and standard deviation of demand, to capture recent level and volatility
  beyond a single lag point. The windows end 48 steps before the target
  (`shift=HORIZON`), for the same reason.

- **Grid-operation variables** (`non_bm_stor`, `pump_storage_pumping` and the six
  interconnector flows) are outcomes of the target half-hour itself, partly driven
  by that half-hour's demand, so they would not be known a day ahead. They are
  replaced by their values 48 steps earlier (e.g. `ifa2_flow_lag_48`), the same
  rule as for the demand lags.

Weather and embedded generation (temperature, embedded wind/solar) are kept as
observed values for the target half-hour. At day-ahead they would in practice be
forecasts, so the models are evaluated under the common "perfect weather forecast"
assumption.

In [17]:
df = create_calendar_features(df)
df = add_cyclical_encoding(df, "hour", period=24)
df = add_cyclical_encoding(df, "day_of_week", period=7)
df = add_cyclical_encoding(df, "month", period=12)

# Day-ahead direct forecasting: only information observed up to t - HORIZON is used to predict t.
# Grid-operation variables are outcomes of the target half-hour itself and would not be known a
# day ahead, so they are replaced by their values HORIZON steps earlier (e.g. ifa2_flow_lag_48)
HORIZON = 48
GRID_COLUMNS = [
    "non_bm_stor", "pump_storage_pumping",
    "ifa_flow", "ifa2_flow", "britned_flow", "moyle_flow", "east_west_flow", "nemo_flow",
]
df = create_day_ahead_features(
    df, horizon=HORIZON, lags=[48, 288, 336], windows=[48, 336], lagged_columns=GRID_COLUMNS
)

In [18]:
df

,settlement_period,nd,embedded_wind_generation,embedded_solar_generation,is_holiday,temperature,day_of_month,day_of_week,day_name,hour,...,nd_rolling_mean_336,nd_rolling_std_336,non_bm_stor_lag_48,pump_storage_pumping_lag_48,ifa_flow_lag_48,ifa2_flow_lag_48,britned_flow_lag_48,moyle_flow_lag_48,east_west_flow_lag_48,nemo_flow_lag_48
settlement_date,,,,,,,,,,,,,,,,,,,,,
2009-01-01 00:00:00+00:00,1,37910,54,0,1,-1.604160,1,3,Thursday,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-01 00:30:00+00:00,2,38047,53,0,1,-1.539581,1,3,Thursday,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-01 01:00:00+00:00,3,37380,53,0,1,-1.475002,1,3,Thursday,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-01 01:30:00+00:00,4,36426,50,0,1,-1.541587,1,3,Thursday,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2009-01-01 02:00:00+00:00,5,35687,50,0,1,-1.608172,1,3,Thursday,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-02-08 21:30:00+00:00,44,30670,3855,0,0,6.090917,8,3,Thursday,21,...,28891.306548,6032.895893,0.0,11.0,2000.0,-4.0,793.0,-424.0,-100.0,969.0
2024-02-08 22:00:00+00:00,45,28684,3873,0,0,6.224113,8,3,Thursday,22,...,28898.431548,6034.146803,0.0,12.0,2000.0,-4.0,867.0,-399.0,-60.0,969.0
2024-02-08 22:30:00+00:00,46,27147,3890,0,0,6.311527,8,3,Thursday,22,...,28905.711310,6033.407344,0.0,14.0,2000.0,-4.0,869.0,-371.0,-136.0,969.0


## Handling Missing Values from Lag/Rolling Features

The lag and rolling features introduce missing values at the start of the series,
where insufficient history exists to compute them (up to 383 steps, i.e. the one-week
window ending 48 steps back). These rows are dropped, representing a negligible loss
given the dataset spans approximately 16 years.

In [19]:
df.isna().sum()

settlement_period                0
nd                               0
embedded_wind_generation         0
embedded_solar_generation        0
is_holiday                       0
temperature                      0
day_of_month                     0
day_of_week                      0
day_name                         0
hour                             0
day_of_year                      0
quarter                          0
month                            0
year                             0
week_of_year                     0
hour_sin                         0
hour_cos                         0
day_of_week_sin                  0
day_of_week_cos                  0
month_sin                        0
month_cos                        0
nd_lag_48                       48
nd_lag_288                     288
nd_lag_336                     336
nd_rolling_mean_48              95
nd_rolling_std_48               95
nd_rolling_mean_336            383
nd_rolling_std_336             383
non_bm_stor_lag_48  

In [20]:
df = df.dropna()

In [21]:
# Full candidate feature set (everything except the target and non-numeric columns).
# `year` is also excluded: the test years lie beyond the training range, where tree models
# can't extrapolate it and MLR/LSTM would extrapolate it unreliably. The demand level is
# already tracked by the lag and rolling features.
PROCESSED_DIR = Path("data/processed")

full_features = [col for col in df.columns if col not in ['nd', 'day_name', 'year']]
df_full = df[full_features + ['nd']]
# Saved in UTC so the timestamps read back cleanly (local time would mix +00:00 and +01:00)
df_full.tz_convert("UTC").to_csv(PROCESSED_DIR / "model_ready_features_day_ahead.csv")

In [22]:
df_full.shape

(264433, 34)

## Final Feature Set for This Notebook

The full engineered feature set has been saved to
`data/processed/model_ready_features_day_ahead.csv`.

It includes all engineered features (calendar, cyclical encodings, lags, rolling statistics, and retained correlation-check columns), restricted to information available
day-ahead (48 steps before each target).
This is used to train both the XGBoost and LSTM models, and to compute SHAP importance from the trained XGBoost model, since SHAP requires visibility into every candidate feature to assess which ones genuinely matter.

## Next Steps

Feature selection by MRFO is not performed in this notebook. In the next
notebook, MRFO will be run twice — once using XGBoost as the fitness function,
once using LSTM — with matched population size and iteration count, so any
difference in the resulting feature subsets can be attributed to the model
rather than to an uneven search budget. Each model will then be trained on its
own matching MRFO selection (XGBoost on the XGBoost-selected features, LSTM on
the LSTM-selected features), alongside both models trained on the full feature
set above.